# Project: 3D Reconstruction from 2D Images
### Studienarbeit - Final Notebook

## 1. Setup and Configuration

### 1.1 Control Switches

**`USE_METADATA`**: 
- **`True`**: Use the `.json` file for exact angles and file order (Recommended for accuracy).
- **`False`**: Guess angles and sort files alphabetically (For baseline comparison).

**`RUN_FULL_RECONSTRUCTION_PIPELINE`**:
- **`True`**: Process files from scratch, run the full reconstruction, and save the result.
- **`False`**: Skip all processing and load the previously saved `.npy` file for quick viewing.

In [ ]:
USE_METADATA = True
RUN_FULL_RECONSTRUCTION_PIPELINE = True

### 1.2 Mount Drive, Clone Repo, and Install Dependencies

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

repo_path = '/content/bohboh'
if not os.path.exists(repo_path):
  !git clone https://github.com/hazempgm/bohboh.git
  %cd {repo_path}
else:
  %cd {repo_path}
  !git pull

!git checkout version_3
!pip install tifffile matplotlib scikit-image tqdm ipywidgets plotly

### 1.3 Imports and Path Definitions

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from skimage.transform import resize
import plotly.graph_objects as go

sys.path.append(os.path.abspath('src'))
from data_loader import load_projection_data
from reconstruction import reconstruct_slice, reconstruct_full_volume

# This directory should contain the .tiff files AND the .json file (if used)
data_dir = '/content/drive/MyDrive/bahbouh/data_bohboh/Images'
output_dir = '/content/drive/MyDrive/bahbouh/'
volume_path = os.path.join(output_dir, 'reconstructed_volume_512.npy')

print("Setup Complete.")

---

## 2. Full Pipeline: From TIFF to 3D Volume

In [ ]:
reconstructed_volume = None
if RUN_FULL_RECONSTRUCTION_PIPELINE:
    # 2.1 Load Data (using the control switch)
    print("--- Step 2.1: Loading Projection Data ---")
    proj_data = load_projection_data(data_dir, use_metadata=USE_METADATA)
    
    if proj_data:
        image_stack, theta = proj_data
        print(f"Data loaded successfully. Image stack shape: {image_stack.shape}, Angles array shape: {theta.shape}")
        
        # 2.2 Visualize a sample projection
        print("\n--- Step 2.2: Visualizing sample projection ---")
        middle_index = image_stack.shape[0] // 2
        plt.figure(figsize=(6, 6)); plt.imshow(image_stack[middle_index], cmap='gray'); plt.title(f'Sample Projection Image (Index #{middle_index})'); plt.show()
        
        # 2.3 Reconstruct and visualize a single test slice
        print("\n--- Step 2.3: Reconstructing and visualizing a test slice ---")
        slice_height_index = image_stack.shape[1] // 2
        sinogram = image_stack[:, slice_height_index, :]
        plt.figure(figsize=(8, 4)); plt.imshow(sinogram, cmap='gray', aspect='auto'); plt.title('Test Sinogram'); plt.xlabel('Detector Position'); plt.ylabel('Projection Angle Index'); plt.show()
        reconstructed_test_slice = reconstruct_slice(sinogram, theta)
        plt.figure(figsize=(6, 6)); plt.imshow(reconstructed_test_slice, cmap='gray'); plt.title('Test Reconstructed Cross-Section'); plt.show()

        # 2.4 Downsample for full reconstruction
        print("\n--- Step 2.4: Downsampling for full reconstruction ---")
        new_shape = (image_stack.shape[0], 512, 512)
        image_stack_small = resize(image_stack, new_shape, anti_aliasing=True)
        print(f"New shape: {image_stack_small.shape}")

        # 2.5 Run full volume reconstruction
        print("\n--- Step 2.5: Running full volume reconstruction ---")
        # We use the corrected function that accepts the 'theta' array
        reconstructed_volume = reconstruct_full_volume(image_stack_small, theta)
        print(f"Final reconstructed volume shape: {reconstructed_volume.shape}")

        # 2.6 Save the final volume
        print("\n--- Step 2.6: Saving final volume ---")
        np.save(volume_path, reconstructed_volume); print(f"File saved to {volume_path}")
    else:
        print("Could not load data. Halting pipeline.")
else:
    print("Skipping full pipeline as requested.")

---

## 3. Load Volume and Visualize

In [ ]:
# If we didn't run the pipeline, we must load the volume from disk
if reconstructed_volume is None:
    if os.path.exists(volume_path):
        print(f"Loading existing volume from {volume_path}...")
        reconstructed_volume = np.load(volume_path)
        print("Volume loaded successfully.")
    else:
        print(f"ERROR: Saved volume not found at {volume_path}.")
        print("Set RUN_FULL_RECONSTRUCTION_PIPELINE = True and re-run to generate it.")

if reconstructed_volume is not None:
    print(f"\nVolume ready for visualization. Shape: {reconstructed_volume.shape}")
else:
    print("\nNo volume data to visualize.")

### 3.1 Interactive 2D Slice Viewers

In [ ]:
def view_slices_2d(volume):
    if volume is None: 
        print("Volume not loaded. Cannot create 2D viewer.")
        return
    def plot_axial(z): plt.figure(figsize=(7,7)); plt.imshow(volume[z,:,:], cmap='gray'); plt.title(f'Axial View (Z={z})'); plt.show()
    def plot_coronal(y): plt.figure(figsize=(7,7)); plt.imshow(volume[:,y,:], cmap='gray'); plt.title(f'Coronal View (Y={y})'); plt.show()
    def plot_sagittal(x): plt.figure(figsize=(7,7)); plt.imshow(volume[:,:,x], cmap='gray'); plt.title(f'Sagittal View (X={x})'); plt.show()
    print("--- Axial Viewer (Top-Down) ---")
    interact(plot_axial, z=widgets.IntSlider(min=0, max=volume.shape[0]-1, value=volume.shape[0]//2))
    print("\n--- Coronal Viewer (Front-Back) ---")
    interact(plot_coronal, y=widgets.IntSlider(min=0, max=volume.shape[1]-1, value=volume.shape[1]//2))
    print("\n--- Sagittal Viewer (Left-Right) ---")
    interact(plot_sagittal, x=widgets.IntSlider(min=0, max=volume.shape[2]-1, value=volume.shape[2]//2))

if reconstructed_volume is not None: view_slices_2d(reconstructed_volume)

### 3.2 Interactive 3D Volume Rendering

In [ ]:
if reconstructed_volume is not None:
    # Downsample for performance
    vis_shape = (128, 128, 128)
    print(f"Downsampling volume for 3D visualization to {vis_shape}...")
    volume_for_3d_vis = resize(reconstructed_volume, vis_shape, anti_aliasing=True)
    
    # Create grid for plotting
    Z, Y, X = volume_for_3d_vis.shape
    x, y, z = np.mgrid[:X, :Y, :Z]
    
    # Determine a good threshold for visibility to hide noise
    vmin = volume_for_3d_vis.min()
    vmax = volume_for_3d_vis.max()
    isomin_threshold = vmin + 0.25 * (vmax - vmin)

    # Create the 3D plot
    fig = go.Figure(data=go.Volume(x=x.flatten(),y=y.flatten(),z=z.flatten(),value=volume_for_3d_vis.flatten(),
        isomin=isomin_threshold, isomax=vmax, opacity=0.1, surface_count=17, 
        caps=dict(x_show=False, y_show=False, z_show=False)))
    
    fig.update_layout(title_text='Interactive 3D Volume Rendering', margin=dict(l=0,r=0,b=0,t=40))
    fig.show()
else:
    print("\nCould not create 3D plot. No volume data available.")

## 4. Post-Processing and 3D Visualization

### 4.1 Segment the Object
We use our automatic thresholding function to create a binary mask that separates the object from the background.

In [ ]:
object_mask = None
if reconstructed_volume is not None:
    object_mask = segment_volume_by_threshold(reconstructed_volume)
else:
    print("No volume to segment.")

### 4.2 Interactive 3D Visualization of Segmented Object
This visualization renders only the voxels that are part of the object mask, revealing the object's true shape.

In [ ]:
if object_mask is not None:
    # Downsample the MASK for performance
    vis_shape = (128, 128, 128)
    print(f"Downsampling mask for 3D visualization to {vis_shape}...")
    mask_for_vis = resize(object_mask, vis_shape, anti_aliasing=False, order=0).astype(float)
    
    # Create grid for plotting
    Z, Y, X = mask_for_vis.shape
    x, y, z = np.mgrid[:X, :Y, :Z]
    
    # Create the 3D plot using the mask
    fig = go.Figure(data=go.Volume(
        x=x.flatten(),
        y=y.flatten(),
        z=z.flatten(),
        value=mask_for_vis.flatten(),
        isomin=0.5, # Show only voxels with value 1 (the object)
        isomax=1.0,
        opacity=0.3, 
        surface_count=10))
    
    fig.update_layout(title_text='Interactive 3D Visualization of Segmented Object', margin=dict(l=0,r=0,b=0,t=40))
    fig.show()
else:
    print("\nCould not create 3D plot. No segmented object mask available.")